# D7 - Structured Logging & Production Debugging

## Objective
Demonstrate centralized structured logging, comparing development plain text logging with production JSON structured logging. Inspect contextual metadata fields, exception tracebacks via `logger.exception()`, and sensitive data masking.

## Concepts Covered
- **Centralized Logging Configuration**: Configuring root and application loggers via `configure_logging()`.
- **Development vs Production Modes**: Human-readable text format vs machine-readable JSON format (`JSONFormatter`).
- **Structured Log Fields & Context**: Emitting structured data (`timestamp`, `level`, `logger`, custom fields).
- **Sensitive Data Masking**: Automatically redacting fields matching `password`, `secret`, `token`, `api_key`, `auth`.
- **Traceback Logging & Duplicate Prevention**: Capturing full stack traces with `logger.exception()` and clearing duplicate handlers.

## Project Implementation
Logging components reside in `app/utils/logging_config.py`:
- `JSONFormatter`: Formats log records into JSON objects with ISO timestamps and sensitive field masking.
- `configure_logging`: Configures console and file loggers, prevents duplicate handlers, and toggles development/production formatters.
- Integrated across `Pipeline`, `Step` classes, and `@timeit` decorators.

## Demonstration
Below, we demonstrate pipeline execution under Development and Production logging modes, capture structured JSON logs in a temporary file, and inspect logged outputs.

In [1]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.utils.logging_config import configure_logging
from app.services.pipeline import Pipeline
from app.services.pipeline_stages import TaskValidationStep, TaskTransformationStep, TaskProcessingStep

# 1. Development Logging Mode Demonstration
print("Configuring Development Plain-Text Logging...")
logger = configure_logging(environment="development")

pipeline = Pipeline([
    TaskValidationStep(),
    TaskTransformationStep(),
    TaskProcessingStep()
])

valid_payload = {"title": "Structured Logging Task", "priority": "MEDIUM", "status": "pending"}
dev_result = pipeline.run(valid_payload)
print(f"Development pipeline execution finished. Status: {dev_result['status']}")

2026-09-22 20:06:19 [INFO] app.utils.cache: Cache miss: Computing configuration for step 'TaskValidationStep' in environment 'production'


2026-09-22 20:06:19 [INFO] app.utils.cache: Cache miss: Computing configuration for step 'TaskTransformationStep' in environment 'production'


2026-09-22 20:06:19 [INFO] app.utils.cache: Cache miss: Computing configuration for step 'TaskProcessingStep' in environment 'production'


2026-09-22 20:06:19 [INFO] app.services.pipeline: Pipeline execution started


2026-09-22 20:06:19 [INFO] app.services.pipeline: Executing pipeline step 'TaskValidationStep'


...

2026-09-22 20:06:19 [INFO] app.services.pipeline_stages: Task processing completed


2026-09-22 20:06:19 [INFO] app.utils.decorators: Function 'process' executed in 0.001149 seconds (1.15 ms)


2026-09-22 20:06:19 [INFO] app.services.pipeline: Completed pipeline step 'TaskProcessingStep'


2026-09-22 20:06:19 [INFO] app.services.pipeline: Pipeline execution completed successfully in 9.89 ms


Configuring Development Plain-Text Logging...
Development pipeline execution finished. Status: PROCESSED


In [2]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import json
import tempfile
from app.utils.logging_config import configure_logging
from app.services.pipeline import Pipeline
from app.services.pipeline_stages import TaskValidationStep, TaskTransformationStep, TaskProcessingStep

# 2. Production JSON Logging Mode Demonstration with File Logging
tmp_file = tempfile.NamedTemporaryFile(suffix=".log", delete=False)
log_file_path = Path(tmp_file.name)
tmp_file.close()

try:
    print(f"Configuring Production JSON Logging (writing to temporary log file)...")
    logger = configure_logging(environment="production", log_file=log_file_path)
    
    pipeline = Pipeline([
        TaskValidationStep(),
        TaskTransformationStep(),
        TaskProcessingStep()
    ])
    
    # Run pipeline with a payload containing sensitive info
    payload_with_secret = {
        "title": "Production Task",
        "priority": "HIGH",
        "status": "pending",
        "api_key": "secret_key_12345"
    }
    
    pipeline.run(payload_with_secret)
    
    # Read generated JSON logs from temporary log file
    log_lines = log_file_path.read_text(encoding="utf-8").strip().splitlines()
    print(f"\nCaptured {len(log_lines)} JSON log entries.")
    
    print("\nSample JSON Log Entry:")
    sample_entry = json.loads(log_lines[0])
    print(json.dumps(sample_entry, indent=2))

finally:
    # Reset logging handlers to close open file handles before unlinking on Windows
    configure_logging(environment="development")
    if log_file_path.exists():
        try:
            log_file_path.unlink()
        except Exception:
            pass

{"timestamp": "2026-09-22T14:36:19.344036+00:00", "level": "INFO", "logger": "app.services.pipeline", "message": "Pipeline execution started", "environment": "production", "event": "pipeline_start", "step_count": 3}


{"timestamp": "2026-09-22T14:36:19.344895+00:00", "level": "INFO", "logger": "app.services.pipeline", "message": "Executing pipeline step 'TaskValidationStep'", "environment": "production", "event": "step_start", "step": "TaskValidationStep"}


{"timestamp": "2026-09-22T14:36:19.345685+00:00", "level": "INFO", "logger": "app.services.pipeline_stages", "message": "Task validation succeeded", "environment": "production", "event": "validation_success", "step": "TaskValidationStep", "record_count": 1}


{"timestamp": "2026-09-22T14:36:19.350995+00:00", "level": "INFO", "logger": "app.services.pipeline", "message": "Completed pipeline step 'TaskValidationStep'", "environment": "production", "event": "step_complete", "step": "TaskValidationStep"}


{"timestamp": "2026-09-22T14:36:19.351976+00:00", "level": "INFO", "logger": "app.services.pipeline", "message": "Executing pipeline step 'TaskTransformationStep'", "environment": "production", "event": "step_start", "step": "TaskTransformationStep"}


...

{"timestamp": "2026-09-22T14:36:19.354644+00:00", "level": "INFO", "logger": "app.services.pipeline_stages", "message": "Task processing completed", "environment": "production", "event": "processing_success", "step": "TaskProcessingStep", "status_label": "PROCESSED", "record_count": 1}


{"timestamp": "2026-09-22T14:36:19.355186+00:00", "level": "INFO", "logger": "app.utils.decorators", "message": "Function 'process' executed in 0.000535 seconds (0.54 ms)", "environment": "production", "event": "function_timing", "function_name": "process", "duration_ms": 0.54}


{"timestamp": "2026-09-22T14:36:19.355797+00:00", "level": "INFO", "logger": "app.services.pipeline", "message": "Completed pipeline step 'TaskProcessingStep'", "environment": "production", "event": "step_complete", "step": "TaskProcessingStep"}


{"timestamp": "2026-09-22T14:36:19.356372+00:00", "level": "INFO", "logger": "app.services.pipeline", "message": "Pipeline execution completed successfully in 12.34 ms", "environment": "production", "event": "pipeline_complete", "duration_ms": 12.34}


Configuring Production JSON Logging (writing to temporary log file)...

Captured 12 JSON log entries.

Sample JSON Log Entry:
{
  "timestamp": "2026-09-22T14:36:19.344036+00:00",
  "level": "INFO",
  "logger": "app.services.pipeline",
  "message": "Pipeline execution started",
  "environment": "production",
  "event": "pipeline_start",
  "step_count": 3
}


In [3]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import json
import tempfile
from app.utils.logging_config import configure_logging
from app.services.pipeline import Pipeline
from app.services.pipeline_stages import TaskValidationStep, TaskTransformationStep, TaskProcessingStep

# 3. Exception Traceback Logging Demonstration
tmp_file = tempfile.NamedTemporaryFile(suffix=".log", delete=False)
log_file_path = Path(tmp_file.name)
tmp_file.close()

try:
    logger = configure_logging(environment="production", log_file=log_file_path)
    pipeline = Pipeline([TaskValidationStep(), TaskTransformationStep(), TaskProcessingStep()])
    
    # Run pipeline with invalid payload to trigger exception logging
    invalid_payload = {"priority": "HIGH"} # Missing title
    
    try:
        pipeline.run(invalid_payload)
    except Exception as e:
        logger.exception("Pipeline execution failed unexpectedly during demonstration", extra={"task_payload": invalid_payload})
        
    log_lines = log_file_path.read_text(encoding="utf-8").strip().splitlines()
    for line in log_lines:
        entry = json.loads(line)
        if "exception" in entry:
            print("\nCaptured Exception Log Entry with Traceback:")
            print(f"Message: {entry['message']}")
            print(f"Exception Snippet: {entry['exception'][:200]}...")
finally:
    # Reset logging handlers to close open file handles before unlinking on Windows
    configure_logging(environment="development")
    if log_file_path.exists():
        try:
            log_file_path.unlink()
        except Exception:
            pass

{"timestamp": "2026-09-22T14:36:19.369451+00:00", "level": "INFO", "logger": "app.services.pipeline", "message": "Pipeline execution started", "environment": "production", "event": "pipeline_start", "step_count": 3}


{"timestamp": "2026-09-22T14:36:19.370422+00:00", "level": "INFO", "logger": "app.services.pipeline", "message": "Executing pipeline step 'TaskValidationStep'", "environment": "production", "event": "step_start", "step": "TaskValidationStep"}


{"timestamp": "2026-09-22T14:36:19.370962+00:00", "level": "WARNING", "logger": "app.services.pipeline_stages", "message": "Task title validation failed: empty or invalid length", "environment": "production", "event": "validation_failure", "step": "TaskValidationStep"}


{"timestamp": "2026-09-22T14:36:19.372237+00:00", "level": "ERROR", "logger": "app.services.pipeline", "message": "Pipeline execution failed at step 'TaskValidationStep': TaskValidationStep: task title is missing or invalid; title is required and must be between 1 and 100 characters.", "environment": "production", "event": "pipeline_failure", "step": "TaskValidationStep", "error_type": "DataValidationError", "error_message": "TaskValidationStep: task title is missing or invalid; title is required and must be between 1 and 100 characters.", "exception": "Traceback (most recent call last):\n  File \"C:\\Users\\Maha Monisha\\OneDrive\\Desktop\\Triton Internship\\task-management-api\\app\\services\\pipeline.py\", line 53, in run\n    current_data = step.process(current_data)\n  File \"C:\\Users\\Maha Monisha\\OneDrive\\Desktop\\Triton Internship\\task-management-api\\app\\services\\pipeline_stages.py\", line 44, in process\n    raise DataValidationError(\n        \"TaskValidationStep: task

{"timestamp": "2026-09-22T14:36:19.374346+00:00", "level": "ERROR", "logger": "app", "message": "Pipeline execution failed unexpectedly during demonstration", "environment": "production", "task_payload": {"priority": "HIGH"}, "exception": "Traceback (most recent call last):\n  File \"C:\\Users\\Maha Monisha\\AppData\\Local\\Temp\\ipykernel_33324\\592833.py\", line 30, in <module>\n    pipeline.run(invalid_payload)\n    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^\n  File \"C:\\Users\\Maha Monisha\\OneDrive\\Desktop\\Triton Internship\\task-management-api\\app\\services\\pipeline.py\", line 53, in run\n    current_data = step.process(current_data)\n  File \"C:\\Users\\Maha Monisha\\OneDrive\\Desktop\\Triton Internship\\task-management-api\\app\\services\\pipeline_stages.py\", line 44, in process\n    raise DataValidationError(\n        \"TaskValidationStep: task title is missing or invalid; title is required and must be between 1 and 100 characters.\"\n    )\napp.utils.exceptions.DataValidationError: 


Captured Exception Log Entry with Traceback:
Message: Pipeline execution failed at step 'TaskValidationStep': TaskValidationStep: task title is missing or invalid; title is required and must be between 1 and 100 characters.
Exception Snippet: Traceback (most recent call last):
  File "C:\Users\Maha Monisha\OneDrive\Desktop\Triton Internship\task-management-api\app\services\pipeline.py", line 53, in run
    current_data = step.process(curre...

Captured Exception Log Entry with Traceback:
Message: Pipeline execution failed unexpectedly during demonstration
Exception Snippet: Traceback (most recent call last):
  File "C:\Users\Maha Monisha\AppData\Local\Temp\ipykernel_33324\592833.py", line 30, in <module>
    pipeline.run(invalid_payload)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^...


## Actual Output
The code cells demonstrate:
1. Readable human format during development mode.
2. Structured JSON formatting in production mode containing ISO timestamps, log levels, logger names, and masked sensitive fields.
3. Complete stack trace capture under the `"exception"` JSON key when an error occurs.

## Key Observations
- JSON formatted logs enable log aggregation tools (e.g. Datadog, ELK stack, CloudWatch) to parse and index log attributes automatically.
- Automatic masking of sensitive fields prevents security credentials from leaking into log files.
- `logger.exception()` captures full exception tracebacks, dramatically simplifying post-mortem production debugging.

## Conclusion
Centralized, structured logging provides complete visibility into application lifecycles, error states, and execution metrics required for production reliability.